## Tools in LLMs

Tools allow an AI model to interact with external systems and perform tasks beyond generating text, such as searching the web, fetching data from a database, calling APIs, or executing code.

Tools are especially important in AI agents, where the model decides when and how to use a tool to accomplish a task.

### Components of a Tool

Tools consist of two main parts:

1. Tool Schema: Defines the tool's name, description, and input arguments, often using a JSON schema. It tells the model what the tool does and what inputs it requires.
2. Tool Implementation: The actual function or coroutine that executes the task when called.

### How Tool Calling Works

1. The developer provides the model with available tool definitions.
2. The model decides whether a tool is needed for the user's request.
3. The model generates a tool call with the required arguments.
4. The application executes the corresponding function.
5. The tool returns its result to the model.
6. The model uses the result to generate a response for the user.

### Example

Suppose an AI assistant has access to a weather tool.

- Tool name: `get_weather`
- Description: Fetches the current weather for a specified city.
- Input: `city = "Gorakhpur"`
- Implementation: Calls a weather API and retrieves the current conditions.
- Output: Weather information that the model can use to answer the user.

Important points:

- The model typically requests a tool call; the application or agent framework executes it.
- A tool call is different from an ordinary text response.
- Tools can connect LLMs to APIs, databases, search engines, and other software.
- Tool calling is a key capability used in agentic AI, but using tools alone does not automatically make an application a fully autonomous agent.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(r"D:\GenAI\.env", override=True)
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")

response = model.invoke("Why do parrots talk?")
response.content

'Parrots don’t “talk” in the way humans do—they don’t understand grammar or abstract meaning the way we do. What they do is **vocal mimicry**, a skill that evolved for very different reasons in the wild. Here’s why they’re capable of copying human speech and why they often do it:\n\n| Reason | How it works in the wild | How it shows up as “talking” with people |\n|--------|------------------------|------------------------------------------|\n| **Social bonding** | Parrots live in flocks where they constantly exchange calls to keep the group together, identify each other, and coordinate activities. A strong, flexible vocal repertoire helps maintain those bonds. | When a parrot is kept as a pet, its human caretakers become its “flock.” The bird learns to imitate the sounds it hears most—especially the voices of the people it spends time with—because those sounds are the most socially relevant cues. |\n| **Territorial and identity signaling** | Many species have distinct “signature calls”

In [25]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the live weather of the given location."""
    return f"The current weather in {location} is sunny."

model_with_tools = model.bind_tools([get_weather])



In [28]:
response = model_with_tools.invoke(["What's the weather like in London?"])
response

AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks for weather in London. We have a function get_weather. We should call it with location "London".', 'tool_calls': [{'id': 'fc_284fee5e-e306-45f1-856a-b4afb54469cf', 'function': {'arguments': '{"location":"London"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 129, 'total_tokens': 181, 'completion_time': 0.108687301, 'completion_tokens_details': {'reasoning_tokens': 25}, 'prompt_time': 0.00586265, 'prompt_tokens_details': None, 'queue_time': 0.41446474, 'total_time': 0.114549951}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c868cf1eaa', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a182f5-e0a6-76bd-88fd-cba83b0b288e-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'London'}, 'id': 'fc_284fee5e-e306-45f1-856a-b4afb54469cf', 'type': 

In [29]:
for tool_call in response.tool_calls:
    # View the tool calls made by the modell
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'location': 'London'}


### Tool Execution Loops

In [35]:
# Step1 : Model generates tool calls
messages = [{"role":"user","content":"What is the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in Boston is currently sunny.


In [36]:
messages

[{'role': 'user', 'content': 'What is the weather in Boston?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "What is the weather in Boston?" Need to get live weather using function get_weather. Use function call.', 'tool_calls': [{'id': 'fc_b2b11085-8514-459c-8f2a-8c23c4ab9136', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 129, 'total_tokens': 182, 'completion_time': 0.11940895, 'completion_tokens_details': {'reasoning_tokens': 26}, 'prompt_time': 0.005597464, 'prompt_tokens_details': None, 'queue_time': 0.307228762, 'total_time': 0.125006414}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_df9620fe21', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a182f8-6be3-77b6-ae95-baec3632c76b-0', tool_calls=[{'name': 'get_weather', 'args': {'location